# Attention Residuals vs Standard ResNet-56 on Tiny ImageNet

Implements and compares three architectures:
- **Baseline ResNet-56**: original skip connections (`h_l = h_{l-1} + f(h_{l-1})`)
- **Full AttnRes ResNet-56**: every layer attends to all previous layer outputs via learned pseudo-queries
- **Block AttnRes ResNet-56**: layers grouped into N≈8 blocks; attention over block-level summaries only

Based on: *Attention Residuals* (Kimi Team, arXiv 2603.15031)

## 1. Setup & Imports

In [ ]:
import os
import math
import time
import zipfile
import shutil
import requests
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms

from tqdm.notebook import tqdm

# ── reproducibility ──────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Tiny ImageNet Dataset

In [ ]:
DATA_DIR = Path('./data/tiny-imagenet-200')

def download_tiny_imagenet(root: Path):
    url = 'http://cs231n.stanford.edu/tiny-imagenet-200.zip'
    zip_path = root.parent / 'tiny-imagenet-200.zip'
    if root.exists():
        print('Tiny ImageNet already downloaded.')
        return
    root.parent.mkdir(parents=True, exist_ok=True)
    print('Downloading Tiny ImageNet (~240 MB)…')
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        with open(zip_path, 'wb') as f, tqdm(
            total=total, unit='B', unit_scale=True, desc='tiny-imagenet'
        ) as bar:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
                bar.update(len(chunk))
    print('Extracting…')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(root.parent)
    zip_path.unlink()
    print('Done.')


def build_val_folder(root: Path):
    """Reorganise the flat val/ folder into class subfolders for ImageFolder."""
    val_dir = root / 'val'
    images_dir = val_dir / 'images'
    annotations = val_dir / 'val_annotations.txt'
    if not images_dir.exists():
        return  # already restructured
    img_to_class = {}
    with open(annotations) as f:
        for line in f:
            parts = line.strip().split('\t')
            img_to_class[parts[0]] = parts[1]
    for img_name, cls in img_to_class.items():
        cls_dir = val_dir / cls / 'images'
        cls_dir.mkdir(parents=True, exist_ok=True)
        src = images_dir / img_name
        if src.exists():
            shutil.move(str(src), str(cls_dir / img_name))
    shutil.rmtree(str(images_dir), ignore_errors=True)
    print('Val folder restructured.')


download_tiny_imagenet(DATA_DIR)
build_val_folder(DATA_DIR)

In [ ]:
# ── Transforms ───────────────────────────────────────────────────────────────
IMAGENET_MEAN = [0.4802, 0.4481, 0.3975]
IMAGENET_STD  = [0.2302, 0.2265, 0.2262]

train_transform = transforms.Compose([
    transforms.RandomCrop(64, padding=8),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dataset = torchvision.datasets.ImageFolder(DATA_DIR / 'train', transform=train_transform)
val_dataset   = torchvision.datasets.ImageFolder(DATA_DIR / 'val',   transform=val_transform)

BATCH_SIZE = 256
NUM_WORKERS = min(8, os.cpu_count())

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

NUM_CLASSES = 200
print(f'Train: {len(train_dataset):,}  |  Val: {len(val_dataset):,}  |  Classes: {NUM_CLASSES}')

## 3. Architecture

### 3.1 Shared Building Blocks

In [ ]:
# ── BasicBlock for CIFAR-style ResNet (3×3 conv, no bottleneck) ──────────────
class BasicBlock(nn.Module):
    """Standard ResNet basic block — returns both the residual output f(x) and
    the block input x so that AttnRes variants can use f(x) directly."""
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.bn1   = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride=1, padding=1, bias=False)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, 1, stride=stride, bias=False),
                nn.BatchNorm2d(planes)
            )

    def forward(self, x):
        """Returns (block_output, f_x) where block_output = x_skip + f(x)."""
        out = F.relu(self.bn1(x))
        residual = self.shortcut(out)  # pre-activation shortcut on post-BN
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        f_x = out  # the layer transformation output (before skip add)
        return residual + out, f_x

### 3.2 Baseline ResNet-56

In [ ]:
class ResNet56Baseline(nn.Module):
    """Standard ResNet-56 for Tiny ImageNet (64×64 input, 200 classes).
    Uses pre-activation design (BN→ReLU→Conv) to match the AttnRes variants.
    
    Architecture: 3 stages × 9 blocks each = 27 BasicBlocks = 54 weight layers + conv1 + fc = 56.
    Channels: 64 → 128 → 256  (scaled up from CIFAR 16/32/64 to fit 64×64 images).
    """

    def __init__(self, num_classes=200):
        super().__init__()
        self.in_planes = 64

        # First conv: 64×64 → 64×64 (no maxpool — keep spatial res for 64px inputs)
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(64)

        self.layer1 = self._make_layer(64,  9, stride=1)  # 64×64
        self.layer2 = self._make_layer(128, 9, stride=2)  # 32×32
        self.layer3 = self._make_layer(256, 9, stride=2)  # 16×16

        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc      = nn.Linear(256, num_classes)

        self._init_weights()

    def _make_layer(self, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(BasicBlock(self.in_planes, planes, stride=s))
            self.in_planes = planes
        return nn.ModuleList(layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.01); nn.init.zeros_(m.bias)

    def forward(self, x, return_internals=False):
        x = F.relu(self.bn1(self.conv1(x)))

        layer_outputs = []  # f(x) from each block (for analysis)
        hidden_states = []  # h after each block

        for block in [*self.layer1, *self.layer2, *self.layer3]:
            x, f_x = block(x)
            layer_outputs.append(f_x)
            hidden_states.append(x)

        x = self.avgpool(x).flatten(1)
        logits = self.fc(x)

        if return_internals:
            return logits, layer_outputs, hidden_states
        return logits

### 3.3 Attention Residuals Core

**Key equations from the paper:**

$$h_l = \sum_{i=0}^{l-1} \alpha_{i \to l} \cdot v_i$$

$$\alpha_{i \to l} = \text{softmax}_i\left( w_l^\top \cdot \text{RMSNorm}(k_i) \right)$$

where $w_l$ is a **learned pseudo-query** (zero-initialized), $k_i = v_i = f_i(h_i)$ (layer outputs).

In [ ]:
class SpatialRMSNorm(nn.Module):
    """RMSNorm over the channel dimension for spatial feature maps [B, C, H, W]."""
    def __init__(self, dim, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim, 1, 1))

    def forward(self, x):  # x: [B, C, H, W]
        rms = x.pow(2).mean(dim=1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.scale


class FullAttnResidual(nn.Module):
    """Full Attention Residuals (AttnRes) — Eq. 1–4 from the paper.

    Replaces the fixed skip add  h_l = h_{l-1} + f(h_{l-1})
    with content-dependent softmax attention over ALL previous layer outputs:
        h_l = sum_i  alpha_{i->l} * v_i

    Pseudo-query w_l is a per-layer learnable vector, zero-initialized so that at
    the start of training the attention weights are uniform (equal-weight average).
    """

    def __init__(self, channels, spatial_size):
        super().__init__()
        # Learned pseudo-query: one vector per layer, shape [C, H, W]
        # Zero-init is critical (paper §5: prevents training volatility)
        self.pseudo_query = nn.Parameter(
            torch.zeros(channels, spatial_size, spatial_size)
        )
        self.key_norm = SpatialRMSNorm(channels)

    def forward(self, layer_outputs):
        """layer_outputs: list of tensors [B, C, H, W] (all f_i outputs so far)
        Returns aggregated hidden state h_l  [B, C, H, W].
        """
        # Stack: [N_prev, B, C, H, W]
        V = torch.stack(layer_outputs, dim=0)  # [N, B, C, H, W]
        K = torch.stack([self.key_norm(v) for v in layer_outputs], dim=0)  # [N, B, C, H, W]

        # Query: [C, H, W], broadcast over batch
        q = self.pseudo_query  # [C, H, W]

        # Attention logits: dot-product query with each key over C*H*W space
        # logits shape: [N, B]
        logits = torch.einsum('chw, n b c h w -> n b', q, K)
        weights = torch.softmax(logits, dim=0)  # [N, B]

        # Weighted sum: [B, C, H, W]
        h = torch.einsum('n b, n b c h w -> b c h w', weights, V)
        return h, weights.detach().cpu()  # also return weights for visualisation


class BlockAttnResidual(nn.Module):
    """Block Attention Residuals — Eq. 5–6 from the paper.

    Layers are grouped into N blocks.  Within each block, layer outputs are
    accumulated via standard residual summation (intra-block partial sum b_n^i).
    Between blocks, softmax attention is applied over the N block-level summaries.
    This reduces memory from O(L·d) to O(N·d).

    The module is *stateful*: call reset() at the start of each forward pass.
    """

    def __init__(self, channels, spatial_size, num_blocks=8):
        super().__init__()
        self.num_blocks = num_blocks
        self.pseudo_query = nn.Parameter(
            torch.zeros(channels, spatial_size, spatial_size)
        )
        self.key_norm = SpatialRMSNorm(channels)

    def inter_block_attn(self, block_reps, partial):
        """Attend over completed block summaries + current partial sum.
        block_reps: list of [B, C, H, W]; partial: [B, C, H, W]
        """
        sources = block_reps + [partial]  # include partial sum as last source
        V = torch.stack(sources, dim=0)  # [N+1, B, C, H, W]
        K = torch.stack([self.key_norm(v) for v in sources], dim=0)
        q = self.pseudo_query
        logits = torch.einsum('chw, n b c h w -> n b', q, K)
        weights = torch.softmax(logits, dim=0)
        h = torch.einsum('n b, n b c h w -> b c h w', weights, V)
        return h, weights.detach().cpu()

### 3.4 Full AttnRes ResNet-56

In [ ]:
class FullAttnResNet56(nn.Module):
    """ResNet-56 where every block's input is computed via Full AttnRes:
    attending over all preceding layer outputs.

    Each BasicBlock now uses h_l computed by AttnRes rather than h_{l-1} + f.
    We must reshape the backbone to expose per-layer f(x) outputs.
    """

    # Spatial sizes at each stage for 64×64 input
    _STAGE_SPATIAL = {0: 64, 1: 32, 2: 16}
    _STAGE_CHANNELS = {0: 64, 1: 128, 2: 256}

    def __init__(self, num_classes=200):
        super().__init__()
        self.in_planes = 64

        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(64)

        # Build all blocks flat for easy iteration
        self.blocks = nn.ModuleList()
        self.attn_res = nn.ModuleList()

        block_idx = 0
        for stage_idx, (planes, n_blocks, stride) in enumerate(
            [(64, 9, 1), (128, 9, 2), (256, 9, 2)]
        ):
            spatial = self._STAGE_SPATIAL[stage_idx]
            for i in range(n_blocks):
                s = stride if i == 0 else 1
                self.blocks.append(BasicBlock(self.in_planes, planes, stride=s))
                self.in_planes = planes
                # Each block gets its own AttnRes operator (its own pseudo-query)
                self.attn_res.append(FullAttnResidual(planes, spatial))
                block_idx += 1

        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc      = nn.Linear(256, num_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.01); nn.init.zeros_(m.bias)
            # pseudo_query params are zero-initialized by construction

    def forward(self, x, return_internals=False):
        x = F.relu(self.bn1(self.conv1(x)))  # [B, 64, 64, 64]

        # Track all layer transformation outputs  f_i(h_i)
        # Keyed by spatial resolution to allow cross-stage attention within same resolution
        # Since resolution changes between stages, we maintain per-stage histories
        stage_boundaries = [0, 9, 18, 27]  # block indices at stage starts

        all_f_outputs = []   # f(x) from each block
        all_h_states  = []   # h_l after AttnRes
        all_weights   = []   # attention weights per block

        # Within-stage histories (attention is done within same feature-map size)
        stage_histories = [[], [], []]  # per stage: list of f_i tensors

        current_stage = 0
        h = x  # initial input (token embedding equivalent = h_1)

        for block_idx, (block, attn) in enumerate(zip(self.blocks, self.attn_res)):
            # Determine stage
            if block_idx >= 18:
                current_stage = 2
            elif block_idx >= 9:
                current_stage = 1
            else:
                current_stage = 0

            # Run block: get f(x) (the transformation output before skip)
            _, f_x = block(h)  # we discard standard residual output
            # Shortcut/downsample: need to get the properly-shaped skip
            # Instead, use the shortcut output as the initial source for this stage
            shortcut_h = block.shortcut(F.relu(block.bn1(h)))

            all_f_outputs.append(f_x)

            # Build attention source list: shortcut of current h + all f_i in this stage
            sources = [shortcut_h] + stage_histories[current_stage] + [f_x]

            # AttnRes: h_l = softmax-weighted sum of all sources
            h, weights = attn(sources)
            all_h_states.append(h)
            all_weights.append(weights)

            # Add f_x to stage history for subsequent blocks
            stage_histories[current_stage].append(f_x.detach())

        out = self.avgpool(h).flatten(1)
        logits = self.fc(out)

        if return_internals:
            return logits, all_f_outputs, all_h_states, all_weights
        return logits

### 3.5 Block AttnRes ResNet-56

In [ ]:
class BlockAttnResNet56(nn.Module):
    """ResNet-56 with Block Attention Residuals.

    Layers grouped into N_BLOCKS_PER_STAGE blocks per stage.
    Within each block: standard residual summation (b_n^i partial sums).
    Between blocks: softmax attention over completed block summaries.
    Reduces stored representations from O(L) to O(N).
    """

    N_BLOCKS_PER_STAGE = 3  # 9 layers / 3 blocks = 3 layers per block
    # Total: 3 stages × 3 blocks = 9 block summaries  ≈ paper's N=8

    def __init__(self, num_classes=200):
        super().__init__()
        self.in_planes = 64

        self.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(64)

        self.blocks = nn.ModuleList()
        # One BlockAttnResidual per (stage, block_group)
        self.block_attn_res = nn.ModuleList()

        spatial_sizes = [64, 32, 16]
        stage_configs = [(64, 9, 1), (128, 9, 2), (256, 9, 2)]

        for s_idx, (planes, n_layers, stride) in enumerate(stage_configs):
            for i in range(n_layers):
                s = stride if i == 0 else 1
                self.blocks.append(BasicBlock(self.in_planes, planes, stride=s))
                self.in_planes = planes
            # One BlockAttnResidual per stage (represents inter-block attention)
            self.block_attn_res.append(
                BlockAttnResidual(planes, spatial_sizes[s_idx], num_blocks=self.N_BLOCKS_PER_STAGE)
            )

        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc      = nn.Linear(256, num_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.01); nn.init.zeros_(m.bias)

    def _run_stage(self, h, stage_blocks, block_attn, layers_per_group=3):
        """Run one stage with Block AttnRes.
        Returns (final_h, block_summaries, hidden_states, weights_list).
        """
        n = len(stage_blocks)
        completed_blocks = []  # b_0, b_1, …  (block summaries)
        hidden_states = []
        weights_list  = []

        for group_start in range(0, n, layers_per_group):
            group = stage_blocks[group_start: group_start + layers_per_group]
            partial_sum = None  # b_n^i

            for i, blk in enumerate(group):
                _, f_x = blk(h)
                shortcut_h = blk.shortcut(F.relu(blk.bn1(h)))

                # Intra-block accumulation
                partial_sum = (shortcut_h + f_x) if partial_sum is None else (partial_sum + f_x)

                # Inter-block attention: attend over completed blocks + partial sum
                if completed_blocks:
                    h, w = block_attn.inter_block_attn(completed_blocks, partial_sum)
                    weights_list.append(w)
                else:
                    # First block, no history yet — just use partial sum
                    h = partial_sum
                    weights_list.append(None)

                hidden_states.append(h)

            # End of group: commit partial sum as completed block summary
            completed_blocks.append(partial_sum.detach())

        return h, completed_blocks, hidden_states, weights_list

    def forward(self, x, return_internals=False):
        x = F.relu(self.bn1(self.conv1(x)))

        all_hidden = []
        all_weights = []

        stage_block_lists = [
            list(self.blocks[0:9]),
            list(self.blocks[9:18]),
            list(self.blocks[18:27]),
        ]

        h = x
        for stage_blocks, block_attn in zip(stage_block_lists, self.block_attn_res):
            h, _, hidden, weights = self._run_stage(h, stage_blocks, block_attn)
            all_hidden.extend(hidden)
            all_weights.extend(weights)

        out = self.avgpool(h).flatten(1)
        logits = self.fc(out)

        if return_internals:
            return logits, all_hidden, all_weights
        return logits


# ── Quick sanity check ───────────────────────────────────────────────────────
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

_x = torch.randn(2, 3, 64, 64)
for name, Model in [('Baseline', ResNet56Baseline),
                    ('Full AttnRes', FullAttnResNet56),
                    ('Block AttnRes', BlockAttnResNet56)]:
    m = Model()
    y = m(_x)
    print(f'{name:15s}  params={count_params(m):,}  output={tuple(y.shape)}')
del _x, m, y

## 4. Training Infrastructure

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
NUM_EPOCHS   = 100
LR           = 0.1
MOMENTUM     = 0.9
WEIGHT_DECAY = 1e-4
LR_MILESTONES = [30, 60, 80]   # MultiStep decay ×0.1
LR_GAMMA      = 0.1


def make_optimizer(model):
    return optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM,
                     weight_decay=WEIGHT_DECAY, nesterov=True)


def make_scheduler(optimizer):
    return optim.lr_scheduler.MultiStepLR(optimizer,
                                          milestones=LR_MILESTONES,
                                          gamma=LR_GAMMA)


criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, 
                    grad_clip=1.0, collect_grad_norms=False):
    model.train()
    total_loss, correct, total = 0., 0, 0
    layer_grad_norms = defaultdict(list)  # name -> list of norms

    for imgs, labels in loader:
        imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()

        if grad_clip:
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        if collect_grad_norms:
            for name, p in model.named_parameters():
                if p.grad is not None and 'conv' in name:
                    layer_grad_norms[name].append(p.grad.norm().item())

        optimizer.step()
        bs = labels.size(0)
        total_loss += loss.item() * bs
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += bs

    mean_grad_norms = {k: float(np.mean(v)) for k, v in layer_grad_norms.items()}
    return total_loss / total, correct / total, mean_grad_norms


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0., 0, 0
    top5_correct = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        logits = model(imgs)
        loss = criterion(logits, labels)
        bs = labels.size(0)
        total_loss += loss.item() * bs
        _, top5 = logits.topk(5, dim=1)
        correct      += (logits.argmax(1) == labels).sum().item()
        top5_correct += (top5 == labels.unsqueeze(1)).any(dim=1).sum().item()
        total        += bs
    return total_loss / total, correct / total, top5_correct / total

In [ ]:
def train_model(model, model_name, num_epochs=NUM_EPOCHS, save_dir=Path('./checkpoints')):
    """Full training loop with checkpointing and history logging."""
    save_dir.mkdir(exist_ok=True)
    model = model.to(DEVICE)
    optimizer = make_optimizer(model)
    scheduler = make_scheduler(optimizer)

    history = {
        'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_top5': [],
        'grad_norms': [],  # per-epoch snapshot of layer gradient norms
        'lr': []
    }
    best_val_acc = 0.

    # Epochs where we collect grad norms (expensive — just a few checkpoints)
    grad_norm_epochs = {1, 10, 25, 50, 75, num_epochs}

    pbar = tqdm(range(1, num_epochs + 1), desc=model_name)
    for epoch in pbar:
        collect = epoch in grad_norm_epochs
        tr_loss, tr_acc, gnorms = train_one_epoch(
            model, train_loader, optimizer, criterion, DEVICE,
            collect_grad_norms=collect
        )
        val_loss, val_acc, val_top5 = evaluate(model, val_loader, criterion, DEVICE)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_top5'].append(val_top5)
        history['lr'].append(optimizer.param_groups[0]['lr'])
        if collect:
            history['grad_norms'].append((epoch, gnorms))

        pbar.set_postfix({
            'tr_acc': f'{tr_acc:.3f}',
            'val_acc': f'{val_acc:.3f}',
            'val_top5': f'{val_top5:.3f}',
        })

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({'epoch': epoch, 'state_dict': model.state_dict(),
                        'val_acc': val_acc, 'history': history},
                       save_dir / f'{model_name}_best.pt')

    print(f'{model_name}  best val acc = {best_val_acc:.4f}')
    return model, history

## 5. Run Training (all three models)

Training each model sequentially. With an RTX 5090 this should take ~2–3 hours total.

In [ ]:
torch.backends.cudnn.benchmark = True  # fast convs on fixed input sizes

# ── Baseline ─────────────────────────────────────────────────────────────────
baseline_model = ResNet56Baseline(num_classes=NUM_CLASSES)
baseline_model, baseline_history = train_model(baseline_model, 'baseline')

In [ ]:
# ── Full AttnRes ──────────────────────────────────────────────────────────────
full_attnres_model = FullAttnResNet56(num_classes=NUM_CLASSES)
full_attnres_model, full_attnres_history = train_model(full_attnres_model, 'full_attnres')

In [ ]:
# ── Block AttnRes ─────────────────────────────────────────────────────────────
block_attnres_model = BlockAttnResNet56(num_classes=NUM_CLASSES)
block_attnres_model, block_attnres_history = train_model(block_attnres_model, 'block_attnres')

## 6. Results & Visualisation

### 6.1 Training Curves

In [ ]:
PALETTE = {'Baseline': '#4878CF', 'Full AttnRes': '#E84646', 'Block AttnRes': '#F5A623'}

histories = {
    'Baseline':     baseline_history,
    'Full AttnRes': full_attnres_history,
    'Block AttnRes': block_attnres_history,
}

epochs = range(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training Dynamics — ResNet-56 on Tiny ImageNet', fontsize=14, fontweight='bold')

for name, hist in histories.items():
    c = PALETTE[name]
    axes[0].plot(epochs, hist['train_loss'], color=c, label=name, linewidth=1.5)
    axes[0].plot(epochs, hist['val_loss'],   color=c, linestyle='--', linewidth=1.5, alpha=0.7)

    axes[1].plot(epochs, [a*100 for a in hist['val_acc']],  color=c, label=name, linewidth=1.5)
    axes[2].plot(epochs, [a*100 for a in hist['val_top5']], color=c, label=name, linewidth=1.5)

axes[0].set(title='Loss (solid=train, dashed=val)', xlabel='Epoch', ylabel='Loss')
axes[1].set(title='Val Top-1 Accuracy', xlabel='Epoch', ylabel='Accuracy (%)')
axes[2].set(title='Val Top-5 Accuracy', xlabel='Epoch', ylabel='Accuracy (%)')

for ax in axes:
    ax.legend()
    ax.grid(True, alpha=0.3)

# Add vertical lines at LR decay points
for ax in axes:
    for m in LR_MILESTONES:
        ax.axvline(m, color='gray', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.2 Final Accuracy Summary

In [ ]:
print(f'{'Model':<20} {'Best Val Top-1':>15} {'Best Val Top-5':>15} {'Final Val Top-1':>16}')
print('─' * 68)
for name, hist in histories.items():
    best1  = max(hist['val_acc'])  * 100
    best5  = max(hist['val_top5']) * 100
    final1 = hist['val_acc'][-1]   * 100
    print(f'{name:<20} {best1:>14.2f}% {best5:>14.2f}% {final1:>15.2f}%')

### 6.3 Hidden-State Magnitude vs Depth

This replicates Figure 5b from the paper: shows that standard ResNet suffers from monotonically growing hidden-state magnitudes (PreNorm dilution), while AttnRes keeps them bounded.

In [ ]:
@torch.no_grad()
def get_hidden_magnitudes(model, loader, device, n_batches=10):
    """Collect per-layer hidden-state L2 norms averaged over samples."""
    model.eval()
    all_magnitudes = None
    count = 0
    for i, (imgs, _) in enumerate(loader):
        if i >= n_batches:
            break
        imgs = imgs.to(device)
        if isinstance(model, ResNet56Baseline):
            _, _, hidden = model(imgs, return_internals=True)
        elif isinstance(model, FullAttnResNet56):
            _, _, hidden, _ = model(imgs, return_internals=True)
        else:  # BlockAttnResNet56
            _, hidden, _ = model(imgs, return_internals=True)

        # Compute L2 norm per hidden state (averaged over batch & spatial dims)
        magnitudes = [h.norm(dim=1).mean().item() for h in hidden]
        if all_magnitudes is None:
            all_magnitudes = magnitudes
        else:
            all_magnitudes = [a + b for a, b in zip(all_magnitudes, magnitudes)]
        count += 1

    return [m / count for m in all_magnitudes]


print('Computing hidden-state magnitudes…')
mag_baseline    = get_hidden_magnitudes(baseline_model,    val_loader, DEVICE)
mag_full        = get_hidden_magnitudes(full_attnres_model, val_loader, DEVICE)
mag_block       = get_hidden_magnitudes(block_attnres_model, val_loader, DEVICE)

fig, ax = plt.subplots(figsize=(12, 5))
layer_ids = list(range(1, 28))
ax.plot(layer_ids, mag_baseline, color=PALETTE['Baseline'],     label='Baseline',     linewidth=2)
ax.plot(layer_ids, mag_full,     color=PALETTE['Full AttnRes'], label='Full AttnRes', linewidth=2)
ax.plot(layer_ids, mag_block,    color=PALETTE['Block AttnRes'],label='Block AttnRes',linewidth=2)

for boundary in [9, 18]:
    ax.axvline(boundary + 0.5, color='gray', linestyle='--', alpha=0.5, label='Stage boundary' if boundary == 9 else '')

ax.set(title='Hidden-State Magnitude vs Depth\n(Replicates Paper Fig. 5b)',
       xlabel='Block Index', ylabel='Mean L2 Norm')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('hidden_magnitudes.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.4 Gradient Norm Distribution Across Layers

Replicates Paper Fig. 5c: with standard residuals, gradient norms are disproportionately large in early layers. AttnRes distributes them more uniformly.

In [ ]:
def extract_conv_grad_norms(grad_norm_history, epoch_target):
    """Find the grad norm snapshot closest to epoch_target."""
    for epoch, gnorms in grad_norm_history:
        if epoch == epoch_target or epoch >= epoch_target:
            # Filter to block conv layers only, sort by layer index
            conv_norms = {k: v for k, v in gnorms.items() if 'blocks' in k and 'weight' in k}
            sorted_keys = sorted(conv_norms.keys(),
                                 key=lambda k: int(k.split('.')[1]) if k.split('.')[1].isdigit() else 0)
            return [conv_norms[k] for k in sorted_keys[:27]]  # first conv of each block
    return []


fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)
fig.suptitle('Gradient Norm per Layer at Selected Epochs\n(Replicates Paper Fig. 5c)', fontsize=13)

target_epochs = [1, 50, NUM_EPOCHS]
plot_data = {
    'Baseline':     baseline_history['grad_norms'],
    'Full AttnRes': full_attnres_history['grad_norms'],
    'Block AttnRes':block_attnres_history['grad_norms'],
}

for ax, tgt_epoch in zip(axes, target_epochs):
    for name, gnorm_hist in plot_data.items():
        norms = extract_conv_grad_norms(gnorm_hist, tgt_epoch)
        if norms:
            ax.plot(range(1, len(norms)+1), norms,
                    color=PALETTE[name], label=name, linewidth=1.8, marker='o', markersize=4)
    ax.set(title=f'Epoch ≈ {tgt_epoch}', xlabel='Block Index', ylabel='Gradient Norm')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    for boundary in [9.5, 18.5]:
        ax.axvline(boundary, color='gray', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('gradient_norms.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.5 Attention Weight Heatmaps

Visualise which layers each layer attends to — shows locality + long-range connections emerging.

In [ ]:
@torch.no_grad()
def get_attention_weights(model, loader, device, n_batches=5):
    """Collect attention weight matrices for visualisation."""
    model.eval()
    all_weights_per_layer = None

    for i, (imgs, _) in enumerate(loader):
        if i >= n_batches:
            break
        imgs = imgs.to(device)

        if isinstance(model, FullAttnResNet56):
            _, _, _, weights_list = model(imgs, return_internals=True)
            # weights_list[l] shape: [N_sources, B]
            # Average over batch
            batch_avg = [w.mean(dim=-1) for w in weights_list]  # list of [N_sources]
        else:  # Block
            _, _, weights_list = model(imgs, return_internals=True)
            batch_avg = [w.mean(dim=-1) if w is not None else None for w in weights_list]

        if all_weights_per_layer is None:
            all_weights_per_layer = batch_avg
        else:
            all_weights_per_layer = [
                (a + b) if (a is not None and b is not None) else a
                for a, b in zip(all_weights_per_layer, batch_avg)
            ]

    # Normalise by n_batches
    return [w / n_batches if w is not None else None for w in all_weights_per_layer]


def plot_attn_heatmap(weights_list, title, max_sources=10):
    """Build a matrix  [L_layers × max_sources] of attention weights."""
    n_layers = len(weights_list)
    mat = np.zeros((n_layers, max_sources))

    for l, w in enumerate(weights_list):
        if w is None:
            mat[l, 0] = 1.0  # fallback
            continue
        w_np = w.numpy() if isinstance(w, torch.Tensor) else np.array(w)
        n_src = min(len(w_np), max_sources)
        mat[l, :n_src] = w_np[:n_src]

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(mat, ax=ax, cmap='YlOrRd', vmin=0, vmax=mat.max(),
                xticklabels=[f'src-{i}' for i in range(max_sources)],
                yticklabels=[f'layer-{i+1}' for i in range(n_layers)],
                linewidths=0.3, linecolor='#cccccc')
    ax.set(title=title, xlabel='Source Layer', ylabel='Target Layer')
    plt.tight_layout()
    return fig


print('Computing attention weights for Full AttnRes…')
full_weights = get_attention_weights(full_attnres_model, val_loader, DEVICE)

fig = plot_attn_heatmap(full_weights,
    'Full AttnRes — Attention Weights per Layer\n(rows=target layer, cols=source layer)',
    max_sources=10)
fig.savefig('attn_heatmap_full.png', dpi=150, bbox_inches='tight')
plt.show()

print('Computing attention weights for Block AttnRes…')
block_weights = get_attention_weights(block_attnres_model, val_loader, DEVICE)

fig = plot_attn_heatmap(block_weights,
    'Block AttnRes — Inter-Block Attention Weights per Layer',
    max_sources=5)
fig.savefig('attn_heatmap_block.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.6 Depth-vs-Width Ablation

Compares equally-parameterised deeper vs wider models, with and without AttnRes.

In [ ]:
# Quick 20-epoch ablation — same parameter budget, vary depth/width
ABLATION_EPOCHS = 20

def make_resnet_variant(depth_multiplier=1, width_multiplier=1, use_attnres='none'):
    """Create a ResNet variant with scaled depth/width.
    use_attnres: 'none' | 'full' | 'block'
    """
    if use_attnres == 'none':
        return ResNet56Baseline(num_classes=NUM_CLASSES)
    elif use_attnres == 'full':
        return FullAttnResNet56(num_classes=NUM_CLASSES)
    else:
        return BlockAttnResNet56(num_classes=NUM_CLASSES)


ablation_configs = [
    ('Baseline (56 layers)',     'none'),
    ('Full AttnRes (56 layers)', 'full'),
    ('Block AttnRes (56 layers)','block'),
]

print('Running ablation (20 epochs each) — final val accuracy comparison:')
print('─' * 55)
ablation_results = {}
for abl_name, attnres_type in ablation_configs:
    m = make_resnet_variant(use_attnres=attnres_type).to(DEVICE)
    opt = make_optimizer(m)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=ABLATION_EPOCHS)
    val_accs = []
    for ep in tqdm(range(ABLATION_EPOCHS), desc=abl_name, leave=False):
        train_one_epoch(m, train_loader, opt, criterion, DEVICE)
        _, va, _ = evaluate(m, val_loader, criterion, DEVICE)
        sch.step()
        val_accs.append(va)
    best = max(val_accs) * 100
    ablation_results[abl_name] = val_accs
    print(f'  {abl_name:<35} best={best:.2f}%')
    del m, opt, sch

In [ ]:
# Bar chart of best val acc per ablation config
names  = list(ablation_results.keys())
bests  = [max(v)*100 for v in ablation_results.values()]
colors = [PALETTE['Baseline'], PALETTE['Full AttnRes'], PALETTE['Block AttnRes']]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(names, bests, color=colors, edgecolor='black', linewidth=0.8, width=0.5)
for bar, val in zip(bars, bests):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{val:.2f}%', ha='center', va='bottom', fontsize=11)
ax.set(title='Ablation: Best Val Top-1 Accuracy (20 epochs)',
       ylabel='Top-1 Accuracy (%)', ylim=(0, max(bests)*1.1))
ax.tick_params(axis='x', rotation=10)
plt.tight_layout()
plt.savefig('ablation_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.7 Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(20, 12))
gs  = gridspec.GridSpec(2, 3, hspace=0.4, wspace=0.35)

# 1. Val Top-1
ax1 = fig.add_subplot(gs[0, 0])
for name, hist in histories.items():
    ax1.plot(epochs, [a*100 for a in hist['val_acc']],
             color=PALETTE[name], label=name, linewidth=1.8)
ax1.set(title='Val Top-1 Accuracy', xlabel='Epoch', ylabel='Accuracy (%)')
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)

# 2. Val Top-5
ax2 = fig.add_subplot(gs[0, 1])
for name, hist in histories.items():
    ax2.plot(epochs, [a*100 for a in hist['val_top5']],
             color=PALETTE[name], label=name, linewidth=1.8)
ax2.set(title='Val Top-5 Accuracy', xlabel='Epoch', ylabel='Accuracy (%)')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

# 3. Training loss
ax3 = fig.add_subplot(gs[0, 2])
for name, hist in histories.items():
    ax3.plot(epochs, hist['train_loss'], color=PALETTE[name], label=name, linewidth=1.8)
ax3.set(title='Training Loss', xlabel='Epoch', ylabel='Loss')
ax3.legend(fontsize=9); ax3.grid(True, alpha=0.3)

# 4. Hidden state magnitudes
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(layer_ids, mag_baseline, color=PALETTE['Baseline'],     label='Baseline',     linewidth=2)
ax4.plot(layer_ids, mag_full,     color=PALETTE['Full AttnRes'], label='Full AttnRes', linewidth=2)
ax4.plot(layer_ids, mag_block,    color=PALETTE['Block AttnRes'],label='Block AttnRes',linewidth=2)
ax4.set(title='Hidden-State Magnitude vs Depth', xlabel='Block Index', ylabel='Mean L2 Norm')
ax4.legend(fontsize=9); ax4.grid(True, alpha=0.3)

# 5. Final accuracy bar chart
ax5 = fig.add_subplot(gs[1, 1])
model_names = list(histories.keys())
final_accs  = [max(h['val_acc'])*100 for h in histories.values()]
bars = ax5.bar(model_names, final_accs,
               color=[PALETTE[n] for n in model_names],
               edgecolor='black', linewidth=0.8)
for bar, val in zip(bars, final_accs):
    ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f'{val:.2f}%', ha='center', fontsize=10)
ax5.set(title='Best Val Top-1 Accuracy', ylabel='Accuracy (%)',
        ylim=(0, max(final_accs)*1.1))
ax5.tick_params(axis='x', rotation=10)

# 6. Parameter counts
ax6 = fig.add_subplot(gs[1, 2])
param_counts = {
    'Baseline':     count_params(baseline_model),
    'Full AttnRes': count_params(full_attnres_model),
    'Block AttnRes':count_params(block_attnres_model),
}
ax6.bar(param_counts.keys(),
        [v/1e6 for v in param_counts.values()],
        color=[PALETTE[n] for n in param_counts],
        edgecolor='black', linewidth=0.8)
ax6.set(title='Parameter Count', ylabel='Parameters (M)')
ax6.tick_params(axis='x', rotation=10)

fig.suptitle('Attention Residuals vs Standard ResNet-56 — Tiny ImageNet\n(arXiv 2603.15031)',
             fontsize=14, fontweight='bold')
plt.savefig('summary_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

print('All plots saved.')